In [0]:
#create variable for storage account
Storage_name = "stytdev001"

#Configure Oauth authentication

spark.conf.set(
    f"fs.azure.account.auth.type.{Storage_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{Storage_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-id"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-secrets"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{Storage_name}.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)


In [0]:
#create Widget for Table_name and Storage_path

dbutils.widgets.text('TABLE_NAME','')
dbutils.widgets.text('STORAGE_PATH','')

TABLE_NAME = dbutils.widgets.get('TABLE_NAME')
STORAGE_PATH = dbutils.widgets.get('STORAGE_PATH')



In [0]:
#create source and sink path

source_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/landingfiles/"
sink_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/bronze/"


# STORAGE_FOLDER_NAME = TABLE_NAME + '_output/'

full_source_path = source_path + STORAGE_PATH + '/' + TABLE_NAME + '.csv'
full_sink_path = sink_path + STORAGE_PATH + '/' + TABLE_NAME

print(full_source_path)
print(full_sink_path)

# display(dbutils.fs.ls(source_path))

abfss://global@stytdev001.dfs.core.windows.net/landingfiles/kolkata_locations_output/kolkata_locations.csv
abfss://global@stytdev001.dfs.core.windows.net/bronze/kolkata_locations_output/kolkata_locations


In [0]:

#read and write data

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(full_source_path)

df_count = df.count()
print(f'with duplicate Data count is : {df_count}')

df_distinct = df.distinct()

df_distinct_count = df_distinct.count()
print(f'without duplicates count is :{df_distinct_count}')

df_distinct.write.format("csv").mode("overwrite").option("header", "true").save(full_sink_path)

print(f'Souce data Count is : {df_count} and destination data count is : {df_distinct_count}')

with duplicate Data count is : 27
without duplicates count is :20
Souce data Count is : 27 and destination data count is : 20
